In [1]:
# ここからスタートする 20251101確認

import win32com.client as win32
import datetime as dt

start_date = dt.datetime(2023, 1, 1, 0, 0, 0)
end_date   = dt.datetime(2026, 5, 31, 23, 59, 59)

PROFILE_NAME = None # 既定でよければ None
STORE_HINTS  = ["kishi_isao@outlook.jp"]

def iter_stores(session):
    """Outlook COMのStoresコレクションを反復可能にする"""
    stores = session.Stores
    for i in range(1, stores.Count + 1):   # 1-based index
        yield stores.Item(i)

def pick_store_by_hints(session, hints):
    if not hints:
        return None
    hints_l = [h.lower() for h in hints]
    for s in iter_stores(session):
        name = (s.DisplayName or "").lower()
        if any(h in name for h in hints_l):
            return s
    return None

# ===== Classic Outlook に接続 =====
app = win32.gencache.EnsureDispatch("Outlook.Application")
session = app.GetNamespace("MAPI")
session.Logon(PROFILE_NAME or "", "", False, False)

# 参考：見つかったストアを一覧表示
print("=== Stores ===")
for s in iter_stores(session):
    try:
        print("-", s.DisplayName)
    except Exception:
        pass

# ストア決定
store = pick_store_by_hints(session, STORE_HINTS) if STORE_HINTS else None
if store is None:
    store = session.DefaultStore

print(f"使用ストア: {store.DisplayName}")

# ストア内の「予定表」取得（olFolderCalendar=9）
try:
    calendar = store.GetDefaultFolder(9)
except Exception:
    root = store.GetRootFolder()
    for cand in ("予定表", "Calendar", "カレンダー"):
        try:
            calendar = root.Folders[cand]
            break
        except Exception:
            pass
    else:
        raise RuntimeError("このストア内で予定表フォルダーが見つかりませんでした。")

items = calendar.Items
items.IncludeRecurrences = True
items.Sort("[Start]")

restriction = (
    f"[Start] >= '{start_date:%m/%d/%Y %I:%M %p}' AND "
    f"[Start] <= '{end_date:%m/%d/%Y %I:%M %p}'"
)
filtered = items.Restrict(restriction)

select_items = []
for it in filtered:
    try:
        _ = it.Start
        select_items.append(it)
    except Exception:
        pass

print("件数:", len(select_items))
for i, ap in enumerate(select_items[:10], 1):
    try:
        print(f"{i:02d}: {ap.Subject}  {ap.Start} → {ap.End}")
    except Exception:
        pass


=== Stores ===
- kishi_isao@outlook.jp
- spax2hr9@diary.ocn.ne.jp
使用ストア: kishi_isao@outlook.jp
件数: 12541
01: 朝食前 体重 血糖値 112/7:04 血圧 118-69/51/7:13, 114-69/52/7:16  2023-01-01 07:00:00+00:00 → 2023-01-01 07:30:00+00:00
02: 朝食　雑煮x2  2023-01-01 07:30:00+00:00 → 2023-01-01 08:00:00+00:00
03: 4分  2023-01-01 00:00:00+00:00 → 2023-01-02 00:00:00+00:00
04: 浜寿司  2023-01-01 11:00:00+00:00 → 2023-01-01 12:00:00+00:00
05: 寿司x10、鍋、おせち料理  2023-01-01 12:30:00+00:00 → 2023-01-01 13:00:00+00:00
06: 初詣  2023-01-01 14:00:00+00:00 → 2023-01-01 16:00:00+00:00
07: 夕食　雑煮x1、おせち料理  2023-01-01 17:30:00+00:00 → 2023-01-01 18:00:00+00:00
08: 朝食前 体重 71.2kg 血糖値 137/7:19 血圧 131-69/55/7:22, 124-71/53/7:24, 123-66/52/7:25   2023-01-02 07:00:00+00:00 → 2023-01-02 07:30:00+00:00
09: 朝食　雑煮x1  2023-01-02 07:30:00+00:00 → 2023-01-02 08:00:00+00:00
10: 飲酒　ウイスキー  2023-01-02 00:00:00+00:00 → 2023-01-03 00:00:00+00:00


In [2]:
#Outlook予定表から取得したデータを食事記録表(Python用).xlsxに転記する
import openpyxl
from openpyxl import utils
from openpyxl import load_workbook
from openpyxl.styles import Alignment
import os

def get_excel_path(filename, folder="ExcelDATA"):
    base = os.path.join(os.environ["OneDrive"], "ドキュメント", "PythonWork")
    return os.path.join(base, folder, filename)

filepath = get_excel_path("食事記録表(--20240806-3).xlsx")

wb=load_workbook(filename=filepath)
ws=wb["食事"]

In [4]:
# 期間を指定してデータを取得　2025/05/31確認

from datetime import datetime, timedelta  # timedeltaもインポートする

def input_date():#日付入力の関数
    while True:  # ユーザーが正しい入力をするまでループ
        try:
            year = int(input("年を入力してください（例: 2025）："))
            month = int(input("月を入力してください（例: 5）："))
            day = int(input("日を入力してください（例: 31）："))
            dt = datetime(year, month, day)
            return dt.date()
        except ValueError as e:
            print("無効な日付です。もう一度入力してください：", e)

def main():
    dt_date = input_date()
    if dt_date and dt_date < datetime(2024, 1, 1).date():
        print("2025年1月20日より前の日付は無効です。")
        return None
    return dt_date

if __name__ == "__main__": # main()を実行する
    dt_date = main()  # main() の戻り値を dt_date に代入, dt_dateは外で使える

    if dt_date:
        print("mainの外で表示：", dt_date.strftime("%Y年%m月%d日"))
    else:
        print("無効な日付が入力されたため、処理できません。")

import datetime
dt_input= dt_date
dt_input=dt_input - datetime.timedelta(days=1)  #予め1日減らしておく,日付はdtetime.date形式
numbers_days=int(input("何日分入力しますか？"))

print("データ取得開始日", dt_input+datetime.timedelta(days=1), "連続取得日数", numbers_days, type(numbers_days))
print("受け渡し日", dt_input)


mainの外で表示： 2026年03月01日
データ取得開始日 2026-03-01 連続取得日数 4 <class 'int'>
受け渡し日 2026-02-28


In [ ]:
24
# 再スタートする場合起動、そうでなければパスしてOK

print("データ取得開始日=", dt_input)
print("連続取得日数", numbers_days)

print("入力した日から再スタートするならｙしないならｎを入力")
y_or_n = input("再スタート") #入力した日から再スタートするならｙしないならｎを入力
if y_or_n=="y":
    dt_input=dt_input-timedelta(days=numbers_days)
    print("データ取得開始日=", dt_input)
    print("連続取得日数", numbers_days)
else:
    print("そのまま続けてください")



In [5]:
# 指定された期間のデータをエクセルファイルに記入する

#食事記録表の転記する日付を探し、そのエクセル上のアドレスを求める

import re

start_date =datetime.date(2023, 1, 1)
for i in range(numbers_days):
    dt_input=dt_input+datetime.timedelta(days=1)

    table_Column_num=1
    table_Row_num=2
    k=0
    Hit_mark=0

    Pick_Date=ws.cell(row=2,column=1).value  #first date of the excel file
    
    #print("1:Pick_Date=", Pick_Date, type(Pick_Date))
    #Pick_Date1=utils.datetime.from_excel(Pick_Date)  #Pick_Dateを年月日時刻に変形する
    #print("2:Pick_Date1=", Pick_Date1, type(Pick_Date1))

    Pick_Date2=Pick_Date.date() #Important!!!!!
    #print("3:Pick_Date2=", Pick_Date2, type(Pick_Date2))
    #print("4:dt_input:", dt_input, "Pick_Date2:", Pick_Date2)
    print(Pick_Date2, dt_input)



    while Pick_Date2!=dt_input:       #Excelの日付データが転記日付と等しくない限り続ける
        k=k+1
        if k>1000:
            break
        else:
            for i in range(11):      #Excelには11日分が7列で1セットになっている
                Pick_Date=ws.cell(row=table_Row_num,column=table_Column_num).value  #日付dataをExcelから読み込む
                
                if isinstance(Pick_Date, int):
                    Pick_Date=utils.datetime.from_excel(Pick_Date)  #Pick_Dateを年月日時刻に変形する
                    
                # print("k:", k, "i:", i, "Pick_Date:", Pick_Date)
                if Pick_Date!=None:
                    #Pick_Date1=utils.datetime.from_excel(Pick_Date)  #Pick_Dateを年月日時刻に変形する
                    Pick_Date2=Pick_Date.date()                     #年月日だけを取り出す
                     #print("k", k, "i", i, Pick_Date2)
                    if Pick_Date2==dt_input:
                        Hit_mark=1
                        print("Hit!")
                        Start_Date=Pick_Date2
                        Start_Row=table_Row_num
                        Start_Column=table_Column_num
                    else:
                        table_Row_num=table_Row_num+7
                else:
                    print("End of Data")
                    break
        table_Row_num=2
        table_Column_num=table_Column_num+7

    if Hit_mark==0:
        print(k)
        print(Pick_Date, Pick_Date2, "S-date is not found")
    else:
        print("dt_input:", dt_input)
        print("Start Date:",Start_Date, type(Start_Date))
        print("Start Row:", Start_Row, type(Start_Row))
        print("Start Column:", Start_Column, type(Start_Column))
        Column_Address=utils.get_column_letter(Start_Column)
        print("Start Column Address:", Column_Address, type(Column_Address))
        

    k=0        
    for select_item in select_items:
        if select_item.Start.date()==Start_Date:
            k=k+1
            print(select_item.Start.date())
            if "朝食　" in select_item.Subject:
                break_fast=select_item.Subject[3:]
                print(break_fast)
                ws.cell(row=Start_Row, column=Start_Column+2).value=break_fast

            if "昼食　" in select_item.Subject:
                lunch=select_item.Subject[3:]
                print(lunch)
                ws.cell(row=Start_Row+1, column=Start_Column+2).value=lunch

            if "夕食　" in select_item.Subject:
                dinner=select_item.Subject[3:]
                print(dinner)
                ws.cell(row=Start_Row+2, column=Start_Column+2).value=dinner

            if "HA1c " in select_item.Subject:
                ha1c=select_item.Subject[5:]
                print("HA1c:", ha1c)
                ws.cell(row=Start_Row+6, column=Start_Column+2).value="定期健診　HA1c"
                ws.cell(row=Start_Row+6, column=Start_Column+4).value=ha1c
                ws.cell(row=Start_Row+6, column=Start_Column+2).font = openpyxl.styles.fonts.Font(color='FF0000')
                ws.cell(row=Start_Row+6, column=Start_Column+4).font = openpyxl.styles.fonts.Font(color='FF0000')
                
            subject = select_item.Subject

            if "服薬" in subject:
                if any(mark in subject for mark in ("〇", "○", "◯")):
                    fukuyaku = "○"
                else:
                    fukuyaku = "×"
                ws.cell(row=Start_Row+4, column=Start_Column+2).value=fukuyaku
                ws.cell(row=Start_Row+4, column=Start_Column+2).alignment = Alignment(horizontal='center')

            if "飲酒" in select_item.Subject:
                #print(select_item.Subject[2:])
                if "◎" in select_item.Subject:
                    insyu="◎"
                    ws.cell(row=Start_Row+4, column=Start_Column+6).value=insyu
                    fill = openpyxl.styles.PatternFill(patternType='solid',fgColor='ADD8E6', bgColor='ADD8E6')
                    ws.cell(row=Start_Row+4, column=Start_Column+6).fill = fill
                    ws.cell(row=Start_Row+4, column=Start_Column+6).alignment = Alignment(horizontal='center')

                    
                else:
                    insyu=select_item.Subject[2:]

                    ws.cell(row=Start_Row+4, column=Start_Column+6).value=insyu
                    fill = openpyxl.styles.PatternFill(patternType='solid',fgColor='FFFFFF', bgColor='FFFFFF')
                    ws.cell(row=Start_Row+4, column=Start_Column+6).fill = fill
                print(insyu)

            # 中程度運動（分）
            if "分" in select_item.Subject:
                undou=select_item.Subject[:-1]
                print(undou)
                if isinstance(undou, int):
                    ws.cell(row=Start_Row+3, column=Start_Column+6).value=undou
                else:
                    undou=int(undou)
                    ws.cell(row=Start_Row+3, column=Start_Column+6).value=undou

            #　運動消費kcal    
            if "運動消費" in select_item.Subject:
                syouhi_kcal=select_item.Subject[5:8]
                print("運動消費", syouhi_kcal)
                ws.cell(row=Start_Row+5, column=Start_Column+6).value=syouhi_kcal
                aaa= ws.cell(row=Start_Row+5, column=Start_Column+6).value
                print("aaa", aaa)
                
            # 歩数
            if "歩数" in select_item.Subject:
                hosu=select_item.Subject[2:8]
                print("歩数", hosu)
                ws.cell(row=Start_Row+5, column=Start_Column+4).value=hosu 
                bbb=ws.cell(row=Start_Row+5, column=Start_Column+4).value
                print("bbb", bbb)

            # 農作業時間
            if "農作業" in select_item.Subject:
                delta=select_item.End-select_item.Start
                delta2=str(delta)
                print(str(delta))
                if delta2[2:4]=="00":
                    delta3=delta2[0:1]+"時間"
                if delta2[2:4]=="15":
                    delta3=delta2[0:1]+".25"+"時間"
                if delta2[2:4]=="30":
                    delta3=delta2[0:1]+".5"+"時間"
                if delta2[2:4]=="45":
                    delta3=delta2[0:1]+".75"+"時間"            

                if select_item.Start.strftime("%p")=="AM":  #午前午後の判定
                    AM_row=Start_Row+6
                    AM_column=Start_Column+3
                    ws.cell(row=AM_row, column=AM_column).value="農作業"+str(delta3)
                    ws.cell(row=AM_row, column=AM_column).font = openpyxl.styles.fonts.Font(color='008B8B')
                if select_item.Start.strftime("%p")=="PM":  #午前午後の判定
                    PM_row=Start_Row+6
                    PM_column=Start_Column+4
                    ws.cell(row=PM_row, column=PM_column).value="農作業"+str(delta3)
                    ws.cell(row=PM_row, column=PM_column).font = openpyxl.styles.fonts.Font(color='008B8B')
                #print("農作業"+str(delta3))

            # ウオーキング、歩数とは別に速足で歩くウオーキングの歩数と距離
            if "ウォーキング" in select_item.Subject:
                walking=select_item.Subject[7:]
                ws.cell(row=Start_Row+3, column=Start_Column+4).value=walking
                #print(walking)
            
            # ゴルフ関係
            golf_excersise=0    

            if "ゴルフ練習" in select_item.Subject:
                golf_excersise=1
                delta=select_item.End-select_item.Start
                delta2=str(delta)

                if delta2[2:4]=="00":
                    delta3=delta2[0:1]+"時間"
                if delta2[2:4]=="30":
                    delta3=delta2[0:1]+".5"+"時間"

                #print(select_item.Start.strftime("%p"))
                if select_item.Start.strftime("%p")=="AM":  #午前午後の判定
                    AM_row=Start_Row+6
                    AM_column=Start_Column+3
                    ws.cell(row=AM_row, column=AM_column).value="ゴルフ練習"+str(delta3)
                    ws.cell(row=AM_row, column=AM_column).font = openpyxl.styles.fonts.Font(color='008B8B')
                if select_item.Start.strftime("%p")=="PM":  #午前午後の判定
                    PM_row=Start_Row+6
                    PM_column=Start_Column+4
                    ws.cell(row=PM_row, column=PM_column).value="ゴルフ練習"+str(delta3)
                    ws.cell(row=PM_row, column=PM_column).font = openpyxl.styles.fonts.Font(color='008B8B')
                #print("ゴルフ練習"+str(delta3))

            if "ゴルフ" in select_item.Subject and golf_excersise==0: 
                golf_play="ゴルフ"+str(select_item.Location)
                #print(golf_play)
                ws.cell(row=Start_Row+6, column=Start_Column+3).value=golf_play
                ws.cell(row=Start_Row+6, column=Start_Column+3).font = openpyxl.styles.fonts.Font(color='008B8B')
                
            if "定期健診" in select_item.Subject: 
                delta=select_item.End-select_item.Start
                delta2=str(delta)
                delta3=delta2[0:1]+"時間"+delta2[2:4]+"分"
                print(select_item.Start.strftime("%p"))
            if "泌尿器科" in select_item.Subject: 
                delta=select_item.End-select_item.Start
                delta2=str(delta)
                delta3=delta2[0:1]+"時間"+delta2[2:4]+"分"
                print(select_item.Start.strftime("%p"))
            if "朝食前" in select_item.Subject:
                #print(select_item.subject)
                A=select_item.Subject.split()            #"朝食前"dataをスペースで分割する-list化する
                print(A, type(A))
                if len(A)!=5:
                    print("LEN", len(A))

                    #体重
                    try:
                        taijyu = float(A[2].replace("kg", ""))
                        ws.cell(row=Start_Row+3, column=Start_Column+2).value = taijyu
                    except ValueError:
                        ws.cell(row=Start_Row+3, column=Start_Column+2).value = None

                    #血糖値

                    import re

                    # 血糖値の位置を探す
                    location_num_B = select_item.Subject.find("血糖値")

                    # 血糖値の部分文字列を抽出（血糖値以降の文字列）
                    blood_text = select_item.Subject[location_num_B:]

                    # 正規表現で「数値/時:分」の形式を抽出
                    match = re.search(r"(\d{2,3}/\d{1,2}:\d{2})", blood_text)

                    if match:
                        kettouti1 = match.group(1)
                        print("kettouti1:", kettouti1)

                        try:
                            value = int(kettouti1.split("/")[0])
                            if value >= 130:
                                ws.cell(row=Start_Row+5, column=Start_Column+2).font = openpyxl.styles.fonts.Font(color='FF0000')
                            elif value < 120:
                                ws.cell(row=Start_Row+5, column=Start_Column+2).font = openpyxl.styles.fonts.Font(color='000000')
                                fill = openpyxl.styles.PatternFill(patternType='solid', fgColor='ADFF2F', bgColor='ADFF2F')
                                ws.cell(row=Start_Row+5, column=Start_Column+2).fill = fill
                        except ValueError:
                            pass
                    else:
                        kettouti1 = None

                    ws.cell(row=Start_Row+5, column=Start_Column+2).value = kettouti1
                  
                    if kettouti1!=None:
                        print("血糖値", kettouti1[:4], type(kettouti1))
                    else:
                        print("血糖値はNone")

                    #血圧
                    if select_item.Subject[-1:]=="," or select_item.Subject[-1:]==" ":
                        nw_select_item=select_item.Subject[-15:-1]
                        ketuatu1=nw_select_item
                        print("NW:", nw_select_item, "K1:", ketuatu1)
                        if ketuatu1!=None:
                            ketuatu=ketuatu1.replace(" ", "")    #スペースの除去
                    else:
                        ketuatu=select_item.Subject[-14:]
                        print(ketuatu, type(ketuatu))
                    ws.cell(row=Start_Row+4, column=Start_Column+4).value=ketuatu
                else:
                    ws.cell(row=Start_Row+3, column=Start_Column+2).value=None
                    ws.cell(row=Start_Row+5, column=Start_Column+2).value=None
                    ws.cell(row=Start_Row+4, column=Start_Column+4).value=None

wb.save(filepath)

2021-12-27 2026-03-01
Hit!
Hit!
Hit!
Hit!
dt_input: 2026-03-01
Start Date: 2026-03-01 <class 'datetime.date'>
Start Row: 51 <class 'int'>
Start Column: 967 <class 'int'>
Start Column Address: AKE <class 'str'>
2026-03-01
073
2026-03-01
運動消費 152
aaa 152
2026-03-01
歩数 07277
bbb 07277
2026-03-01
2026-03-01
['朝食前', '体重', '71.2kg', '血糖値', '129/7:58', '血圧', '127-76/57/7:51,', '127-70/59/7:53,', '115-71/54/7:55'] <class 'list'>
LEN 9
kettouti1: 129/7:58
血糖値 129/ <class 'str'>
115-71/54/7:55 <class 'str'>
2026-03-01
カレーライス100g、茹でブロッコリー
2026-03-01
1:30:00
2026-03-01
2026-03-01
雑煮x1.5
2026-03-01
1:30:00
2026-03-01
2026-03-01
ご飯150g、スープ、焼肉、バナナ、カステラ
2021-12-27 2026-03-02
Hit!
Hit!
Hit!
dt_input: 2026-03-02
Start Date: 2026-03-02 <class 'datetime.date'>
Start Row: 58 <class 'int'>
Start Column: 967 <class 'int'>
Start Column Address: AKE <class 'str'>
2026-03-02
2026-03-02
◎
2026-03-02
029
2026-03-02
運動消費 154
aaa 154
2026-03-02
歩数 06660
bbb 06660
2026-03-02
2026-03-02
['朝食前', '体重', '70.4kg', '血糖値',

In [ ]:
# とりあえず使わない
# H:\Python\ExcelDATA\食事記録表(--20240806).xlsxのファイルはいっぱいになり、データの追加が不可能になったので、
# ”食事”sheetの1921年12月26日以前を削除しH:\Python\ExcelDATA\食事記録表(--20240806-2).xlsxを作成した。20250104

# プログラムの修正用のプログラム修正後はOutlook_to_Excel2にコピーして使用すること
#　2024/06/23 確認済
# 日付入力しOutlookの予定表(spax2hr9@diary.ocn.ne.jp)からその日のデータを取得する(20240623確認)

#予定表データをOutlookから取得する(時間がかかるので先に実施しておく)
import win32com.client
import datetime
#from datetime import datetime

outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
#root_folder = outlook.Folders.Item(1)    #Folders.Item(1)でkishi_isao@outlook.jpを指定

#calender = root_folder.Folders['予定表']  #Folders['予定表']で予定表のデータを取り込む
recipient = outlook.CreateRecipient("kishi_isao@outlook.jp")
calender = outlook.GetSharedDefaultFolder(recipient, 9)  # 9は予定表


# calender = outlook.GetDefaultFolder(9)

items = calender.Items # このitemsが一つ一つの「予定」

select_items = [] # 指定した期間内の予定を入れるリスト

#あらかじめ期間を指定しておく（例えば、2023/01/01以降）
# 予定を抜き出したい期間を指定
start_date =datetime.date(2023, 1, 1)
end_date =datetime.date(2025, 11, 30)
for item in items:
    if start_date <= item.start.date() <= end_date:
        select_items.append(item)

# import win32com.client as win32
# import datetime as dt

# # ===== 取得対象の期間を指定（必要に応じて変更）=====
# start_date = dt.datetime(2023, 1, 1, 0, 0, 0)
# end_date   = dt.datetime(2025, 12, 31, 23, 59, 59)

# # ===== Outlook 接続 =====
# ns = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")

# # 既定の予定表（最も確実。複数アカウントでも既定に紐づく）
# calendar = ns.GetDefaultFolder(9)  # 9 = olFolderCalendar（予定表）

# # ===== アイテムを高速抽出（再発予定を含む）=====
# items = calendar.Items
# items.IncludeRecurrences = True
# items.Sort("[Start]")

# # Restrict は米国式 12時間表記で指定する必要あり
# restriction = (
#     f"[Start] >= '{start_date:%m/%d/%Y %I:%M %p}' AND "
#     f"[Start] <= '{end_date:%m/%d/%Y %I:%M %p}'"
# )

# filtered = items.Restrict(restriction)

# # 元コードの仕様に合わせて「指定期間の予定を select_items に詰める」
# select_items = [item for item in filtered]

# print("件数:", len(select_items))
# # 例: 最初の数件だけ確認
# for i, ap in enumerate(select_items[:5], 1):
#     try:
#         print(f"{i:02d}: {ap.Subject}  {ap.Start} → {ap.End}")
#     except Exception:
#         pass


In [ ]:
import win32com.client.gencache as gc
print(gc.GetGeneratePath())
